<a href="https://colab.research.google.com/github/Yeeejj/EMHA/blob/feat/per-task-v3/notebooks/colab_per_task_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EMHA v3: per-exercise training (EXPLORATORY, Deviation 6)

Bootstrap only: this notebook clones the branch, installs, sets paths, and calls
modules (CLAUDE.md). Pipeline logic, model code, and every hyperparameter live
in `src/` (`src/training/run_per_task_v3.py`, `src/utils/config.py`); never add
them here. If `pip` upgrades torch, restart the runtime and run the cells again
from the top. Plan: `DOCS/plans/EMHA_V3_MASTER_PLAN.txt`, phases 3-10.

Needs the Colab secret `GITHUB_TOKEN` with notebook access switched on.

**Modes** (cell 1, `MODE`):
- `"snapshot"`: SHA-256 of `results/` and `models/` under `DRIVE_OUTPUT_ROOT`,
  written to `SNAPSHOT_FILE` (`python -m scripts.hash_tree snapshot`). Refuses
  if `SNAPSHOT_FILE` already exists. Run once, before any v3 step (step 3.1).
- `"compare"`: the same tree checked against `SNAPSHOT_FILE`; new files are
  allowed only under the v3 paths (`ALLOW_NEW`). Must exit 0 (step 10.1).
- `"run"`: `python -m src.training.run_per_task_v3 --step STEP`. The dataset
  zip (`DRIVE_ZIP`) is unzipped to the local disk (`LOCAL_DATA_ROOT`); outputs
  and a log (`logs/v3_<STEP>_<UTC time>.log`) go to `DRIVE_OUTPUT_ROOT`.

Steps in order: preflight, lr, cnn, hmm, combine, evaluate, permutation-lr,
draws, permutation-cnn, report. Finished (cell, fold) units and draws are
cached on Drive, so after a disconnect, **Run all with the same parameters**.
Do not open any v3 output before the report step (step 9.4).

**Draws are checked on the Drive server** (`scripts/drive_check.py`). With
`STEP = "draws"`, the mount cell also signs in to the Drive API (one pop-up),
and the runner is called once per draw in `DRAWS`. After each draw the Drive
API waits up to 5 minutes for the file on the server and uploads it itself if
it is still missing (`drive server: perm_<k>.csv confirmed` or `uploaded`), so
a session that dies loses at most the draw in progress. At the end the draws
are copied to `DRAWS_STAGING` on the local disk, Drive is flushed, and the
whole folder is checked again (`drive server: N/20 present`, also in
`logs/v3_draws_<UTC time>_drive_check.log`). If a draw is not confirmed, the
cell stops with an error and keeps the runtime, so the local copies survive.
Disjoint `DRAWS` ranges may run in parallel sessions (each from its own copy
of this notebook: File -> Save a copy in Drive).

**Every run ends with a flush of Drive.** With `AUTO_DISCONNECT = False` the
run cell still calls `drive.flush_and_unmount()`, so outputs written through
the mount reach Drive before you disconnect; check them on Drive, then
disconnect. Keep free space on Drive: when the quota is full, the mount drops
uploads without an error.

**Pay As You Go has no background execution:** keep this browser tab open and
the PC awake for the whole session.

In [ ]:
MODE = "run"  # "run" | "snapshot" | "compare"
STEP = "preflight"  # preflight | lr | cnn | hmm | combine | evaluate | permutation-lr | draws | permutation-cnn | report
CELLS = None  # e.g. "D1" or "D1,D2"; None = all 24
FOLDS = None  # e.g. "0"; None = all 5
DRAWS = None  # e.g. "0-4" (STEP = "draws" only)
N_JOBS = -1
DEVICE = "cuda"  # "cpu" for CPU-only steps
REQUIRE_GPU = None  # e.g. "A100"; None = no check
AUTO_DISCONNECT = True
BRANCH = "feat/per-task-v3"
DRIVE_ZIP = "/content/drive/MyDrive/EMHA/emha-crops.zip"
LOCAL_DATA_ROOT = "/content/emha-crops"
DRIVE_OUTPUT_ROOT = "/content/drive/MyDrive/EMHA/run_outputs"
SNAPSHOT_FILE = DRIVE_OUTPUT_ROOT + "/v3_checksums_before.txt"
TOPS = ["results", "models"]
ALLOW_NEW = [
    "results/per_task_v3/",
    "models/cnn_v3/",
    "models/hmm_v3/",
    "results/final/permutation_runs/primary_v3/",
    "results/final/EXPLORATORY_v3.txt",
    "results/final/fig_v3_",
    "results/final/permutation_null_v3.png",
]

REPO = "Yeeejj/EMHA"
REPO_DIR = "/content/EMHA"

In [ ]:
from google.colab import auth, drive

drive.mount("/content/drive")
if MODE == "run" and STEP == "draws":
    auth.authenticate_user()  # Drive API, for the server-side draw check

In [ ]:
from google.colab import userdata

token = userdata.get("GITHUB_TOKEN")

import base64
import os
import subprocess

if os.path.isdir(REPO_DIR):
    print(f"{REPO_DIR} already exists; using it as is")
else:
    auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    subprocess.run(
        ["git", "-c", f"http.extraHeader=Authorization: Basic {auth}",
         "clone", "--branch", BRANCH, f"https://github.com/{REPO}.git", REPO_DIR],
        check=True,
    )
    del auth
del token
os.chdir(REPO_DIR)
branch = subprocess.run(["git", "rev-parse", "--abbrev-ref", "HEAD"],
                        capture_output=True, text=True, check=True).stdout.strip()
if branch != BRANCH:
    raise SystemExit(f"{REPO_DIR} is on {branch!r}, not {BRANCH}; delete it and Run all")

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
!git rev-parse HEAD
!git describe --tags --always
!git tag --list protocol-frozen
!nproc
import torch

print("torch", torch.__version__, "| CUDA build", torch.version.cuda,
      "| CUDA available", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
if MODE == "run" and REQUIRE_GPU:
    gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU"
    if REQUIRE_GPU not in gpu:
        raise SystemExit(
            f"GPU is {gpu!r}, not {REQUIRE_GPU}: Runtime -> Change runtime type -> "
            f"{REQUIRE_GPU}, then Run all again (or set REQUIRE_GPU = None)"
        )
    print("GPU check passed:", gpu)

In [ ]:
import shutil

if MODE == "run":
    labels_csv = os.path.join(LOCAL_DATA_ROOT, "metadata", "labels.csv")
    if not os.path.isfile(labels_csv):
        if not os.path.isfile(DRIVE_ZIP):
            raise SystemExit(f"{DRIVE_ZIP} not found; upload emha-crops.zip to Drive")
        partial = LOCAL_DATA_ROOT + ".partial"
        shutil.rmtree(partial, ignore_errors=True)
        print("unzipping", DRIVE_ZIP, "->", LOCAL_DATA_ROOT)
        shutil.unpack_archive(DRIVE_ZIP, partial, "zip")
        os.rename(partial, LOCAL_DATA_ROOT)
    if not os.path.isfile(labels_csv):
        raise SystemExit(f"{LOCAL_DATA_ROOT} has no metadata/labels.csv; check the zip")
    print(open(os.path.join(LOCAL_DATA_ROOT, "MANIFEST.txt")).read())

In [ ]:
THREAD_ENV = {"OMP_NUM_THREADS": "1", "MKL_NUM_THREADS": "1", "OPENBLAS_NUM_THREADS": "1"}
os.makedirs(os.path.join(DRIVE_OUTPUT_ROOT, "logs"), exist_ok=True)
if MODE == "run":
    os.environ["EMHA_DATA_ROOT"] = LOCAL_DATA_ROOT
    os.environ["EMHA_OUTPUT_ROOT"] = DRIVE_OUTPUT_ROOT
    os.environ.update(THREAD_ENV)
    print({k: os.environ[k] for k in ("EMHA_DATA_ROOT", "EMHA_OUTPUT_ROOT", *THREAD_ENV)})
print({"MODE": MODE, "STEP": STEP, "CELLS": CELLS, "FOLDS": FOLDS, "DRAWS": DRAWS,
       "N_JOBS": N_JOBS, "DEVICE": DEVICE})

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, REPO_DIR)
from scripts.drive_check import (
    check_draws, confirm_draw, drive_folder_id, drive_service, drive_upload,
)

DRAWS_STAGING = "/content/v3_draws_staging"
MY_DRIVE = "/content/drive/MyDrive"

In [ ]:
import datetime
import shutil
import sys

if MODE == "run" and STEP == "draws":
    probe = ("import sys; "
             "from src.analysis.permutation import parse_draws; "
             "from src.training.run_per_task_v3 import runs_dir; "
             "from src.utils.config import config; "
             "n = config.per_task_v3.n_cnn_permutations; "
             "print(runs_dir(config.training.seed, config)); print(n); "
             "print(*parse_draws(sys.argv[1] or f'0-{n - 1}', n))")
    found = subprocess.run([sys.executable, "-c", probe, DRAWS or ""], cwd=REPO_DIR,
                           capture_output=True, text=True, check=True).stdout
    draws_dir, n_draws, draw_list = found.strip().splitlines()[-3:]
    draw_list = [int(k) for k in draw_list.split()]
    if not Path(draws_dir).is_relative_to(MY_DRIVE):
        raise SystemExit(f"{draws_dir} is not under {MY_DRIVE}")
    draws_rel = Path(draws_dir).relative_to(MY_DRIVE).as_posix()
    print("draws folder:", draws_dir, "| n =", n_draws, "| this session:", draw_list)
    svc = drive_service()

if MODE == "snapshot":
    if os.path.exists(SNAPSHOT_FILE):
        raise SystemExit(f"{SNAPSHOT_FILE} already exists; snapshots are never overwritten")
    cmd = [sys.executable, "-u", "-m", "scripts.hash_tree", "snapshot", DRIVE_OUTPUT_ROOT,
           "--tops", *TOPS, "--out", SNAPSHOT_FILE]
elif MODE == "compare":
    cmd = [sys.executable, "-u", "-m", "scripts.hash_tree", "compare", DRIVE_OUTPUT_ROOT,
           "--before", SNAPSHOT_FILE, "--tops", *TOPS, "--allow-new", *ALLOW_NEW]
elif MODE == "run":
    cmd = [sys.executable, "-u", "-m", "src.training.run_per_task_v3", "--step", STEP,
           "--device", DEVICE, "--n-jobs", str(N_JOBS)]
    if CELLS:
        cmd += ["--cells", CELLS]
    if FOLDS:
        cmd += ["--folds", FOLDS]
    if DRAWS and STEP != "draws":
        cmd += ["--draws", DRAWS]
else:
    raise SystemExit(f"MODE must be run, snapshot or compare, not {MODE!r}")
# draws: one runner call per draw, each confirmed on the Drive server before the next
cmds = ([cmd + ["--draws", str(k)] for k in draw_list]
        if MODE == "run" and STEP == "draws" else [cmd])

head = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO_DIR,
                      capture_output=True, text=True, check=True).stdout.strip()
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
name = STEP if MODE == "run" else MODE
log_path = os.path.join(DRIVE_OUTPUT_ROOT, "logs", f"v3_{name}_{stamp}.log")
print("log:", log_path)
draws_ok = True
draws_folder = None
with open(log_path, "a", buffering=1) as log:
    log.write(f"git HEAD {head} | GPU {gpu} | nproc {os.cpu_count()}\n")
    for one in cmds:
        log.write("command: " + " ".join(one) + "\n")
        proc = subprocess.Popen(
            one, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1,
        )
        for line in proc.stdout:
            print(line, end="")
            log.write(line)
        run_exit = proc.wait()
        if run_exit != 0 or not (MODE == "run" and STEP == "draws"):
            break
        draw_file = Path(draws_dir) / f"perm_{int(one[-1]):04d}.csv"
        if draws_folder is None:
            draws_folder = drive_folder_id(svc, draws_rel)
        if draw_file.is_file():
            check_line, draws_ok = confirm_draw(svc, draws_folder, draw_file)
        else:
            check_line, draws_ok = f"drive server: {draw_file.name} missing locally", False
        print(check_line)
        log.write(check_line + "\n")
        if not draws_ok:
            break
    log.write(f"{name} exit code: {run_exit}\n")
    print(f"{name} exit code:", run_exit)
    if MODE == "run" and STEP == "draws":
        shutil.rmtree(DRAWS_STAGING, ignore_errors=True)
        if os.path.isdir(draws_dir):
            shutil.copytree(draws_dir, DRAWS_STAGING, ignore=shutil.ignore_patterns("*.tmp"))
        else:
            os.makedirs(DRAWS_STAGING)

mounted = True
if MODE == "run" and STEP == "draws":
    drive.flush_and_unmount()
    mounted = False
    check_lines, final_ok = check_draws(svc, DRAWS_STAGING, draws_rel, n_draws)
    draws_ok = draws_ok and final_ok
    print("\n".join(check_lines))
    report = Path("/content") / f"v3_{name}_{stamp}_drive_check.log"
    report.write_text("\n".join(check_lines) + "\n")
    logs_rel = (Path(DRIVE_OUTPUT_ROOT) / "logs").relative_to(MY_DRIVE).as_posix()
    drive_upload(svc, drive_folder_id(svc, logs_rel), report, "text/plain")
    print("Drive unmounted; Run all mounts it again for the next batch")
if mounted and not AUTO_DISCONNECT:
    drive.flush_and_unmount()
    mounted = False
    print("Drive flushed and unmounted; check the outputs on Drive before disconnecting")
if not draws_ok:
    raise SystemExit(
        f"draws not confirmed on the Drive server; runtime kept so the copies in "
        f"{DRAWS_STAGING} survive. Do not disconnect; report this before going on."
    )
if AUTO_DISCONNECT:
    print("releasing runtime")
    if mounted:
        drive.flush_and_unmount()
    from google.colab import runtime

    runtime.unassign()
elif run_exit != 0:
    raise SystemExit(f"{name} failed (exit {run_exit})")